# Imports

In [ ]:
import pandas as pd
import numpy as np
import json

# Load Data

In [ ]:
#df = pd.read_sas("../data/LLCP2025XPT/LLCP2025.XPT", format="xport", encoding="latin-1")
#df.to_parquet("../data/brfss2025.parquet")
#df_full = pd.read_parquet("../data/brfss2025.parquet")
#df_il = df[df["_STATE"] == 17]
#df_il.to_parquet("../data/brfss2025_il.parquet")

# Clean Data

In [ ]:
target = "DIABETE4"
filter_cols = ["DIABTYPE"]        # used only to drop type 1, never as a predictor

cat_predictors_raw = [
    "GENHLTH", "_METSTAT", "DIFFWALK", "ADDEPEV3", "CVDSTRK3", "MEDCOST1",
    "_HLTHPL2", "_TOTINDA",
    "_RFHYPE6", "_RFCHOL3", "_MICHD", "_ASTHMS1", "_DRDXAR2",
    "_SEX", "_EDUCAG", "_INCOMG1", "_RACE",
    "_SMOKER3", "_CURECI3",
]
num_predictors_raw = [
    "SLEPTIM1", "PHYSHLTH", "MENTHLTH",
    "_AGE80", "HTM4", "_BMI5",
    "_DRNKWK3",
    "FTJUDA2_", "FRUTDA2_", "GRENDA1_", "FRNCHDA_", "POTADA1_","VEGEDA2_", 
]
predictors = cat_predictors_raw + num_predictors_raw
assert len(set(predictors)) == len(predictors), "duplicate column in list"

df_raw = pd.read_parquet(
    "../data/brfss2025_il.parquet",
    columns=[target] + filter_cols + predictors,
)

df = df_raw[
    df_raw["DIABETE4"].isin([1, 2, 3, 4])       # Remove people who did not know or refused
    & ~df_raw["DIABTYPE"].isin([1, 9])          # Remove people with type 1 or refused
    & ~((df_raw["DIABETE4"] == 1) & df_raw["DIABTYPE"].isna())      # Remove people with diabetes but a missing type
].copy()
df["DIABETES"] = (df["DIABETE4"] == 1).astype(int)      # Create binary target variable for people with type 2 diabetes

# Check positive response with raw values
display(df["DIABETES"].value_counts(dropna=False))
display(df_raw.loc[df_raw["DIABETE4"] == 1, "DIABTYPE"].value_counts(dropna=False))

# Rename columns
rename_map = {
    # categorical
    "GENHLTH":  "gen_health",
    "_METSTAT": "metro_status",
    "DIFFWALK": "diff_walk",
    "ADDEPEV3": "depression",
    "CVDSTRK3": "stroke_history",
    "MEDCOST1": "cost_barrier",
    "_HLTHPL2": "has_insurance",
    "_TOTINDA": "phys_active_30",
    "_RFHYPE6": "high_bp",
    "_RFCHOL3": "high_chol",
    "_MICHD":   "heart_disease",
    "_ASTHMS1": "asthma_status",
    "_DRDXAR2": "arthritis",
    "_SEX":     "sex",
    "_EDUCAG":  "education",
    "_INCOMG1": "income",
    "_RACE":    "race",
    "_SMOKER3": "smoker_status",
    "_CURECI3": "ecig_user",
    # numeric
    "SLEPTIM1": "sleep_time",
    "PHYSHLTH": "poor_phys_days",
    "MENTHLTH": "poor_ment_days",
    "_AGE80":   "age",
    "HTM4":     "height_m",
    "_BMI5":    "bmi",
    "_DRNKWK3": "drinks_per_week",
    "FTJUDA2_": "fruit_juice_daily",
    "FRUTDA2_": "fruit_daily",
    "GRENDA1_": "green_veg_daily",
    "FRNCHDA_": "fries_daily",
    "POTADA1_": "potatoes_daily",
    "VEGEDA2_": "other_veg_daily",
}

df = df.rename(columns=rename_map)

# Recreate predictor lists
cat_predictors = [rename_map[c] for c in cat_predictors_raw]
num_predictors = [rename_map[c] for c in num_predictors_raw]

# Rescale columns (They had two implied decimal places)
scale_cols = [
    "height_m", "bmi", "drinks_per_week", "fruit_juice_daily", "fruit_daily", "green_veg_daily", "fries_daily", "potatoes_daily", "other_veg_daily"
]
df[scale_cols] = df[scale_cols] / 100

# Fix categorical values
df.loc[df["gen_health"] >= 7, "gen_health"] = np.nan            # Recode don't know and refused as NaN
df.loc[df["diff_walk"] >= 7, "diff_walk"] = np.nan              # Recode don't know and refused as NaN
df.loc[df["depression"] >= 7, "depression"] = np.nan            # Recode don't know and refused as NaN
df.loc[df["stroke_history"] >= 7, "stroke_history"] = np.nan    # Recode don't know and refused as NaN
df.loc[df["cost_barrier"] >= 7, "cost_barrier"] = np.nan        # Recode don't know and refused as NaN
df.loc[df["has_insurance"] == 9, "has_insurance"] = np.nan      # Recode don't know and refused as NaN
df.loc[df["phys_active_30"] == 9, "phys_active_30"] = np.nan    # Recode don't know and refused as NaN
df.loc[df["high_bp"] == 9, "high_bp"] = np.nan                  # Recode don't know and refused as NaN
df["high_chol"] = df["high_chol"].fillna(9)                     # NaN values were people who haven't had it checked, include with don't know and refused
df.loc[df["asthma_status"] == 9, "asthma_status"] = np.nan      # Recode don't know and refused as NaN
df.loc[df["education"] == 9, "education"] = np.nan              # Recode don't know and refused as NaN
df.loc[df["race"] == 9, "race"] = np.nan                        # Recode don't know and refused as NaN
df.loc[df["smoker_status"] == 9, "smoker_status"] = np.nan      # Recode don't know and refused as NaN
df.loc[df["ecig_user"] == 9, "ecig_user"] = np.nan              # Recode don't know and refused as NaN

# Recode cat columns
yes_no = {1: 1, 2: 0}   # variable where 1 = yes, 2 = no
no_yes = {1: 0, 2: 1}   # variable where 1 = no, 2 = yes

recode_map = {
    # Binary
    "diff_walk": yes_no,
    "depression": yes_no,
    "stroke_history": yes_no,
    "cost_barrier": yes_no,
    "heart_disease": yes_no,
    "arthritis": yes_no,
    "has_insurance": yes_no,
    "phys_active_30": yes_no,
    "high_bp": no_yes,
    "ecig_user": no_yes,
    # Multi-class
    "sex": {1: "male", 2: "female"},
    "metro_status": {1: "metro", 2: "non-metro"},
    "high_chol": {1: "no", 2: "yes", 9: "unknown"},
    "asthma_status": {1: "current", 2: "former", 3: "never"},
    "smoker_status": {1: "daily", 2: "some_days", 3: "former", 4: "never"},
    "race": {1: "white", 2: "black", 3: "other", 4: "asian", 5: "other", 6: "other", 7: "multiracial", 8: "hispanic"},  # Categories with <100 are set to other (American Indian/Alaska Native / Native Hawaiian / other)
    "gen_health": {1: "excellent", 2: "very_good", 3: "good", 4: "fair", 5: "poor"},
    "education": {1: "no_hs", 2: "hs_grad", 3: "some_college", 4: "college_grad"},
    "income": {1: "under_15k", 2: "15-25k", 3: "25-35k", 4: "35-50k", 5: "50-100k", 6: "100-200k", 7: "over_200k", 9: "unknown"},
}

for col, m in recode_map.items():
    n_missing = df[col].isna().sum()
    df[col] = df[col].map(m)
    assert df[col].isna().sum() == n_missing, f"{col}: unmapped code became NaN"

df["gen_health"] = pd.Categorical(df["gen_health"], categories=["excellent", "very_good", "good", "fair", "poor"], ordered=True)
df["education"] = pd.Categorical(df["education"], categories=["no_hs", "hs_grad", "some_college", "college_grad"], ordered=True)

# Fix numerical values
num_cols = df.select_dtypes("number").columns
df[num_cols] = df[num_cols].mask(df[num_cols].abs() < 1e-70, 0)     # Fix zero bug

df["sleep_time"] = df["sleep_time"].replace({77: np.nan, 99: np.nan})       # Recode don't know and refused as NaN
df["sleep_cat"] = pd.cut(
    df["sleep_time"],
    bins=[0, 6, 8, 24],
    labels=["short", "normal", "long"],     # <=6 hours, 7-8 hours, >=9 hours
)
df["sleep_cat"] = df["sleep_cat"].cat.reorder_categories(["normal", "short", "long"])       # Makes normal the reference category
num_predictors.remove("sleep_time")
cat_predictors.append("sleep_cat")

df["poor_phys_days"] = df["poor_phys_days"].replace({88: 0, 77: np.nan, 99: np.nan})        # Recode don't know and refused as NaN
df["poor_ment_days"] = df["poor_ment_days"].replace({88: 0, 77: np.nan, 99: np.nan})        # Recode don't know and refused as NaN

df.loc[df["drinks_per_week"] > 300, "drinks_per_week"] = np.nan     # Remove impossible values and people who did not answer, binge drink column was used to confirm extreme values
df.loc[df["fruit_daily"] > 16, "fruit_daily"] = np.nan              # Use CDC cutoff for impossible values
df.loc[df["fruit_juice_daily"] > 16, "fruit_juice_daily"] = np.nan  # Use CDC cutoff for impossible values
df.loc[df["fries_daily"] > 19, "fries_daily"] = np.nan              # Use CDC cutoff for impossible values
df.loc[df["other_veg_daily"] > 23, "other_veg_daily"] = np.nan      # Use CDC cutoff for impossible values
df.loc[df["green_veg_daily"] > 23, "green_veg_daily"] = np.nan      # Use CDC cutoff for impossible values
df.loc[df["potatoes_daily"] > 23, "potatoes_daily"] = np.nan        # Use CDC cutoff for impossible values

veg_parts = ["green_veg_daily", "other_veg_daily", "potatoes_daily"]
df["veg_daily"] = df[veg_parts].sum(axis=1, skipna=False).round(2)  # Create new column for total daily vegetable intake
df.loc[df["veg_daily"] > 23, "veg_daily"] = np.nan                  # Use CDC cutoff for impossible values

num_predictors = [c for c in num_predictors if c not in veg_parts] + ["veg_daily"]      # Add veg_daily and remove parts

Column Notes:
- 
- many num columns may need log transform to care for outlier values

# Save to new file

In [ ]:
cols = ["DIABETES"] + cat_predictors + num_predictors
df[cols].to_parquet("../data/brfss2025_il_clean.parquet")

with open("../data/brfss2025_il_clean_columns.json", "w") as f:
    json.dump({
    "target": "DIABETES",
    "cat_predictors": cat_predictors,
    "num_predictors": num_predictors
    }, f, indent=2
    )

# Missingness

In [ ]:
predictors_final = cat_predictors + num_predictors
print(df[predictors_final].dropna().shape[0] / len(df))     # share of complete rows

missing = (df.isna().mean().sort_values() * 100).round(2)   # Percentage of missing values for each column
missing[predictors_final].sort_values(ascending=False)

Diabetes split between complete and incomplete rows 
- Even split for target
- Dropping would still lose a lot of positive rows
- Imputation is still likely best route

In [ ]:
complete = df[predictors_final].notna().all(axis=1)
df.groupby(complete)["DIABETES"].mean()